In [1]:
import os
import jax

# NOTE: Eric, this this chunk below, it will use your compilation cache and speed things up if you restart and need to run the exact same code (assuming no JIT recompilation)
cache_dir = os.path.join(os.getcwd(), ".jax_cache")
if not os.path.exists(cache_dir):
    os.makedirs(cache_dir)
jax.config.update("jax_compilation_cache_dir", cache_dir)
jax.config.update("jax_persistent_cache_min_entry_size_bytes", -1)
jax.config.update("jax_persistent_cache_min_compile_time_secs", 0)
jax.experimental.compilation_cache.compilation_cache.set_cache_dir(cache_dir)

import json
import time
import jax.numpy as jnp
from jax.random import key as jkey

import numpy as np
import pickle
from tqdm import tqdm

from tensorflow_probability.substrates.jax import distributions as tfd

from genparticles.datatypes import *
from genparticles.model_3d import *
from genparticles.inference import *
from genparticles.dataloader import *
from genparticles.utils import *
from genparticles.evaluation import *

model_jsimulate = jax.jit(HDGMM_model_3d.simulate)
model_jimportance = jax.jit(HDGMM_model_3d.importance)

######################################################
## CONFIGURATION START
######################################################

EXPERIMENT_NAME = "Davis_Benchmark_Results"
# Define the experiment save directory
EXPERIMENT_SAVE_DIR = os.path.join(os.getcwd(), "temp_results")
os.makedirs(EXPERIMENT_SAVE_DIR, exist_ok=True)

print(f"Experiment save directory: {EXPERIMENT_SAVE_DIR}")

HYPERPARAMS = {
    "number_of_blobs": 500,
    "number_of_hyperblobs": 9,
    "num_seeded_runs": 1,
}

SAVE_DATA = True

# Video configuration
VIDEO_NAME = "breakdance"  # Change this to any video name (e.g., "bear", "blackswan", "goat", etc.)

# List of stimuli to run - just blackswan for now
# List of stimuli to run - use VIDEO_NAME
DAVIS_STIMULI = [VIDEO_NAME]

#SET PATH TO STIMULI DATA
DAVIS_STIMULI_PATH = "/home/esli/BADJA/DAVIS/davis_3D_motions"
# DAVIS_STIMULI_PATH = "/home/esli/GenParticles_neural_stimulus/assets/genparticles_CV_benchmark_data/genparticles_davis_preprocessed"
DAVIS_SEGMASKS_PATH = "/home/esli/BADJA/DAVIS/Annotations/Full-Resolution"
# DAVIS_SEGMASKS_PATH = "/home/esli/GenParticles_neural_stimulus/assets/genparticles_CV_benchmark_data/davis_segmasks"

######################################################
## CONFIGURATION END
######################################################


def run_hdgmm_benchmark(stimulus, hyperparams, experiment_name, experiment_dir):
    """
    Run HDGMM benchmark for a single stimulus with multiple random seeds.
    
    Args:
        stimulus: Name of the stimulus to process
        hyperparams: Dictionary containing hyperparameters for the experiment
        experiment_name: Name of the experiment
        experiment_dir: Directory where experiment data should be saved
        
    Returns:
        tuple: (tracking_data, random_seeds, img_dims) for the stimulus
    """
    # Expand hyperparameters from the dictionary
    number_of_blobs = hyperparams["number_of_blobs"]
    number_of_hyperblobs = hyperparams["number_of_hyperblobs"]
    num_seeded_runs = hyperparams["num_seeded_runs"]
    
    print(f"\nStarting benchmark for stimulus: {stimulus}")

    # Extract feature point data
    tracked_points, tracked_motion_vectors, num_data_tsteps, img_dims = extract_3d_points_and_motion_vectors_data(DAVIS_STIMULI_PATH, stimulus)
    first_frame_seg = get_segmentation_mask(stimulus, 0, DAVIS_SEGMASKS_PATH, img_dims=img_dims, flatten=True)

    kmeans_chm, roi_blob_indices, roi_hyperblob_indices = make_hierarchical_kmeans_chm_with_mask_fixed_hyperblob(tracked_points, number_of_blobs, number_of_hyperblobs, segmentation_mask = first_frame_seg, motion_vectors = tracked_motion_vectors)

    # Set up Gibbs sampling dials
    GIBBS_DIALS = {
        "blob_weights": True,
        "hyperblob_weights": False,
        "blob_assignments": True,
        "hyperblob_assignments": False,
        "hyperblob_covs": False,
        "blob_covs": True,
        "blob_vel_covs": True,
        "blob_vel_means": True,
        "hyperblob_means": False,
        "blob_means": True,
        'hyperblob_rot_vels': False,
        'hyperblob_trans_vels': False
    }

    # hyperparams are scaled to the length scale of the data
    num_datapoints = kmeans_chm['datapoints', 'datapoint_positions'].shape[0]
    num_blobs = kmeans_chm['blobs', 'hyperblob_assignments'].shape[0]
    num_hyperblobs = kmeans_chm['hyperblobs', 'hyperblob_means'].shape[0]
    empirical_mu_H = jnp.median(kmeans_chm['datapoints', 'datapoint_positions'], axis = 0)
    empirical_sigma_H = (10*0.5)**2
    empirical_Psi_B = jnp.median(kmeans_chm['blobs', 'blob_covs'][roi_blob_indices], axis = 0)
    empirical_Psi_H = jnp.median(kmeans_chm['hyperblobs', 'hyperblob_covs'][roi_hyperblob_indices], axis = 0)
    empirical_Psi_V = jnp.median(kmeans_chm['blobs', 'blob_vel_covs'][roi_blob_indices], axis = 0)
    mean_blobs_per_roi_hyperblob = jnp.sum(jnp.isin(kmeans_chm['blobs', 'hyperblob_assignments'], roi_hyperblob_indices)) / len(roi_hyperblob_indices)
    empirical_nu_H = f_(int(mean_blobs_per_roi_hyperblob))
    mean_points_per_roi_blob = jnp.sum(jnp.isin(kmeans_chm['datapoints', 'blob_assignments'], roi_blob_indices)) / len(roi_blob_indices)
    empirical_nu_B = empirical_nu_V = f_(int(mean_points_per_roi_blob))

    # Create hyperparameters
    hypers = HDGMM_Hyperparams.create(
        outlier_prob=f_(5e0),
        outlier_velocity_gamma_shape=f_(5.0),
        outlier_velocity_gamma_rate=f_(1.0),
        alpha=f_(1.0),
        beta=f_(1.0),
        mu_H=empirical_mu_H,
        sigma_H=empirical_sigma_H,
        nu_H=empirical_nu_H,
        Psi_H=empirical_Psi_H,
        nu_B=empirical_nu_B,
        Psi_B=empirical_Psi_B,
        sigma_V=f_(10e14),
        nu_V=empirical_nu_V,
        Psi_V=empirical_Psi_V,
        translation_gaussian_scale=snp(f_(0.2)),
        translation_max_radius=snp(0.35),
        translation_num_radii_cells=snp(15),
        translation_theta_step_deg=snp(15),
        rotation_vmf_kappa=snp(f_(100)),
        rotation_angle_max_deg=snp(25),
        rotation_angle_step_deg=snp(0.375),
        n_hyperblobs=num_hyperblobs,
        n_blobs=num_blobs,
        n_datapoints=num_datapoints,
    )

    # Initialize lists to store results
    multiple_tracking_data = []

    # Run multiple seeds
    for j in range(num_seeded_runs):
        print(f"Running seed {j+1} of {num_seeded_runs} for {stimulus}")

        key = jkey(np.random.randint(1e9))

        # Initialize model
        key, key_importance = jax.random.split(key)
        init_tr, _ = model_jimportance(key_importance, kmeans_chm, (hypers,))
        init_hdgmm_state = init_tr.get_retval()

        # Run initial Gibbs sweeps
        key, init_gibbs_key = jax.random.split(key)
        gibbs_wtrs = hdgmm_full_gibbs(init_gibbs_key, init_hdgmm_state, 15, GIBBS_DIALS, use_weighted_blobs=True, num_gibbs_inner_loops=1)

        # Get final state and clear memory
        init_hdgmm_state = gibbs_wtrs[-1].retval
        
        # Run tracking
        key, tracking_key = jax.random.split(key)
        tracking_wtrs = hdgmm_tracking_gibbs(tracking_key, init_hdgmm_state, tracked_points, tracked_motion_vectors)
        # Clear memory
        
        # Extract trace variables for visualization and evaluation
        tracking_data = []
        for frame_idx in range(len(tracking_wtrs)):
            # Extract comprehensive trace data from each frame
            frame = tracking_wtrs[frame_idx]
            frame_data = {
                # Hyperparameters
                'n_blobs': frame.retval.hypers.n_blobs,
                'n_hyperblobs': frame.retval.hypers.n_hyperblobs,
                'n_datapoints': frame.retval.hypers.n_datapoints,
                
                # Datapoints state
                'blob_assignments': np.array(frame.retval.datapoints_state.blob_assignments),
                'datapoint_positions': np.array(frame.retval.datapoints_state.datapoint_positions),
                'datapoint_vels': np.array(frame.retval.datapoints_state.datapoint_vels),
                
                # Blobs state
                'blob_weights': np.array(frame.retval.blobs_state.blob_weights),
                'blob_means': np.array(frame.retval.blobs_state.blob_means),
                'blob_covs': np.array(frame.retval.blobs_state.blob_covs),
                'blob_vel_means': np.array(frame.retval.blobs_state.blob_vel_means),
                'blob_vel_covs': np.array(frame.retval.blobs_state.blob_vel_covs),
                'hyperblob_assignments': np.array(frame.retval.blobs_state.hyperblob_assignments),
                
                # Hyperblobs state
                'hyperblob_weights': np.array(frame.retval.hyperblobs_state.hyperblob_weights),
                'hyperblob_means': np.array(frame.retval.hyperblobs_state.hyperblob_means),
                'hyperblob_trans_vels': np.array(frame.retval.hyperblobs_state.hyperblob_trans_vels),
                'hyperblob_rot_vels': np.array(frame.retval.hyperblobs_state.hyperblob_rot_vels),
            }
            tracking_data.append(frame_data)
        
        # Store tracking results
        multiple_tracking_data.append(tracking_data)
        

    return multiple_tracking_data, img_dims

# Process each stimulus and collect results
all_results = {}
experiment_start_time = time.time()
for i, stimulus in enumerate(DAVIS_STIMULI):
    print(f"\nProcessing stimulus {i+1}/{len(DAVIS_STIMULI)}: {stimulus}")

    multiple_tracking_data, img_dims = run_hdgmm_benchmark(stimulus, HYPERPARAMS, EXPERIMENT_NAME, EXPERIMENT_SAVE_DIR)
    all_results[stimulus] = multiple_tracking_data


experiment_end_time = time.time()
total_minutes = (experiment_end_time - experiment_start_time) / 60
print(f"Experiment completed in {total_minutes:.2f} minutes for {len(DAVIS_STIMULI)} stimuli")

# NOTE: Eric, don't run this old eval code

# # Run evaluation and visualization after all stimuli are processed
# evaluate_tracking_results(
#     all_results, 
#     annotations_path=DAVIS_SEGMASKS_PATH,
#     save_data=SAVE_DATA,
#     output_dir=os.path.join(EXPERIMENT_SAVE_DIR, EXPERIMENT_NAME),
#     experiment_name=EXPERIMENT_NAME
# )


Experiment save directory: /home/esli/GenParticles_NeurIPS/temp_results

Processing stimulus 1/1: breakdance

Starting benchmark for stimulus: breakdance
Running seed 1 of 1 for breakdance
Experiment completed in 2.62 minutes for 1 stimuli


In [2]:
# run this new eval code

scene_to_run = VIDEO_NAME # by default

experiment_metrics, best_visualization_data = evaluate_single_davis_video(
    davis_name = scene_to_run,
    multiple_genparticles_list = all_results[scene_to_run],
    annotations_path = DAVIS_SEGMASKS_PATH,
    counting_threshold=100,
    img_dims = img_dims,
    fps_list=None,
    render_results_video=True,
    experiment_save_dir=None
)


📊 Processing dataset: breakdance

📈 Trial Results for breakdance:
Trial    Recall (%)   Precision  FPR      IoU      AUC ROC 
--------------------------------------------------------------
1        84.28        0.658      0.045    0.585    0.061   
--------------------------------------------------------------
Mean     84.28        0.658      0.045    0.585    0.061   

🎬 Creating results video (with particle overlay)...


In [3]:
# edit the experiment_save_dir path to what you want

# this might take a while

# if you dont want to save to disk, set experiment_save_dir = None, it should render here

create_genmatter_results_video(best_visualization_data, annotations_path = DAVIS_SEGMASKS_PATH, img_dims = img_dims, experiment_save_dir = '.')

💾 Saving visualization to: ./breakdance_results_video.mp4
✅ Visualization complete!


### Arijit: the code below is untouched

In [ ]:
# Comprehensive Visualization of Trace Variables for Camel Experiment
import matplotlib.pyplot as plt
from PIL import Image
import numpy as np
from mpl_toolkits.mplot3d import Axes3D

# Get the camel results
stimulus = VIDEO_NAME
if stimulus in all_results:
    print(f"Visualizing trace variables for: {stimulus}")
    
    tracking_data = all_results[stimulus][0]  # First seed run
    num_frames = len(tracking_data)
    
    # ===== 1. BLOB STRUCTURE OVER TIME =====
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    fig.suptitle(f'{stimulus.capitalize()} Tracking - Blob Structure Over Time', fontsize=16, fontweight='bold')
    
    # 1a. Number of blobs and datapoints per frame
    num_blobs = [frame['n_blobs'] for frame in tracking_data]
    num_datapoints = [frame['n_datapoints'] for frame in tracking_data]
    
    axes[0, 0].plot(range(num_frames), num_blobs, marker='o', linewidth=2, color='blue', label='Blobs')
    axes[0, 0].plot(range(num_frames), num_datapoints, marker='s', linewidth=2, color='green', label='Datapoints')
    axes[0, 0].set_xlabel('Frame Number', fontsize=11)
    axes[0, 0].set_ylabel('Count', fontsize=11)
    axes[0, 0].set_title('Number of Blobs and Datapoints per Frame')
    axes[0, 0].legend()
    axes[0, 0].grid(True, alpha=0.3)
    
    # 1b. Blob weights over time (for first 10 blobs)
    for blob_idx in range(min(10, tracking_data[0]['n_blobs'])):
        blob_weights_over_time = [frame['blob_weights'][blob_idx] if blob_idx < len(frame['blob_weights']) else 0 
                                   for frame in tracking_data]
        axes[0, 1].plot(range(num_frames), blob_weights_over_time, linewidth=1.5, alpha=0.7, label=f'Blob {blob_idx}')
    axes[0, 1].set_xlabel('Frame Number', fontsize=11)
    axes[0, 1].set_ylabel('Blob Weight', fontsize=11)
    axes[0, 1].set_title('Blob Weights Over Time (First 10 Blobs)')
    axes[0, 1].grid(True, alpha=0.3)
    axes[0, 1].legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=8)
    
    # 1c. Distribution of blob assignments
    blob_assignment_counts = []
    for frame in tracking_data:
        unique, counts = np.unique(frame['blob_assignments'], return_counts=True)
        blob_assignment_counts.append(len(counts))
    
    axes[1, 0].plot(range(num_frames), blob_assignment_counts, marker='o', linewidth=2, color='purple')
    axes[1, 0].set_xlabel('Frame Number', fontsize=11)
    axes[1, 0].set_ylabel('Number of Active Blobs', fontsize=11)
    axes[1, 0].set_title('Number of Blobs with Assigned Datapoints')
    axes[1, 0].grid(True, alpha=0.3)
    
    # 1d. Average blob velocity magnitude over time
    avg_vel_magnitudes = []
    for frame in tracking_data:
        vel_means = frame['blob_vel_means']
        magnitudes = np.linalg.norm(vel_means, axis=1)
        avg_vel_magnitudes.append(np.mean(magnitudes))
    
    axes[1, 1].plot(range(num_frames), avg_vel_magnitudes, marker='o', linewidth=2, color='red')
    axes[1, 1].set_xlabel('Frame Number', fontsize=11)
    axes[1, 1].set_ylabel('Average Velocity Magnitude', fontsize=11)
    axes[1, 1].set_title('Average Blob Velocity Magnitude Over Time')
    axes[1, 1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    # ===== 2. 3D BLOB POSITIONS =====
    fig = plt.figure(figsize=(16, 6))
    fig.suptitle(f'{stimulus.capitalize()} Tracking - 3D Blob Positions', fontsize=16, fontweight='bold')
    
    # Show blob positions for frame 0, middle frame, and last frame
    frames_to_show = [0, num_frames // 2, num_frames - 1]
    
    for i, frame_idx in enumerate(frames_to_show):
        ax = fig.add_subplot(1, 3, i + 1, projection='3d')
        
        blob_means = tracking_data[frame_idx]['blob_means']
        blob_weights = tracking_data[frame_idx]['blob_weights']
        
        # Scale point sizes by blob weights
        sizes = blob_weights * 1000
        scatter = ax.scatter(blob_means[:, 0], blob_means[:, 1], blob_means[:, 2], 
                           c=range(len(blob_means)), s=sizes, alpha=0.6, cmap='viridis')
        
        ax.set_xlabel('X', fontsize=10)
        ax.set_ylabel('Y', fontsize=10)
        ax.set_zlabel('Z', fontsize=10)
        ax.set_title(f'Frame {frame_idx}: {len(blob_means)} Blobs')
        
    plt.tight_layout()
    plt.show()
    
    # ===== 3. HYPERBLOB STRUCTURE =====
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    fig.suptitle(f'{stimulus.capitalize()} Tracking - Hyperblob Structure', fontsize=16, fontweight='bold')
    
    # 3a. Hyperblob weights over time
    for hyperblob_idx in range(tracking_data[0]['n_hyperblobs']):
        hyperblob_weights_over_time = [frame['hyperblob_weights'][hyperblob_idx] 
                                        for frame in tracking_data]
        axes[0].plot(range(num_frames), hyperblob_weights_over_time, linewidth=2, 
                    alpha=0.7, marker='o', label=f'Hyperblob {hyperblob_idx}')
    axes[0].set_xlabel('Frame Number', fontsize=11)
    axes[0].set_ylabel('Hyperblob Weight', fontsize=11)
    axes[0].set_title('Hyperblob Weights Over Time')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)
    
    # 3b. Number of blobs per hyperblob (first frame)
    hyperblob_assignments = tracking_data[0]['hyperblob_assignments']
    unique, counts = np.unique(hyperblob_assignments, return_counts=True)
    axes[1].bar(unique, counts, color='skyblue', edgecolor='black')
    axes[1].set_xlabel('Hyperblob Index', fontsize=11)
    axes[1].set_ylabel('Number of Blobs', fontsize=11)
    axes[1].set_title('Number of Blobs per Hyperblob (Frame 0)')
    axes[1].grid(True, alpha=0.3, axis='y')
    
    plt.tight_layout()
    plt.show()
    
    # ===== 4. DATAPOINT STATISTICS =====
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    fig.suptitle(f'{stimulus.capitalize()} Tracking - Datapoint Statistics', fontsize=16, fontweight='bold')
    
    # 4a. Average datapoint velocity magnitude
    avg_datapoint_vel = []
    for frame in tracking_data:
        vel_magnitudes = np.linalg.norm(frame['datapoint_vels'], axis=1)
        avg_datapoint_vel.append(np.mean(vel_magnitudes))
    
    axes[0, 0].plot(range(num_frames), avg_datapoint_vel, marker='o', linewidth=2, color='orange')
    axes[0, 0].set_xlabel('Frame Number', fontsize=11)
    axes[0, 0].set_ylabel('Average Velocity Magnitude', fontsize=11)
    axes[0, 0].set_title('Average Datapoint Velocity Magnitude')
    axes[0, 0].grid(True, alpha=0.3)
    
    # 4b. Datapoint positions spread (std dev)
    position_spread = []
    for frame in tracking_data:
        std_devs = np.std(frame['datapoint_positions'], axis=0)
        position_spread.append(np.mean(std_devs))
    
    axes[0, 1].plot(range(num_frames), position_spread, marker='o', linewidth=2, color='green')
    axes[0, 1].set_xlabel('Frame Number', fontsize=11)
    axes[0, 1].set_ylabel('Average Position Std Dev', fontsize=11)
    axes[0, 1].set_title('Datapoint Position Spread Over Time')
    axes[0, 1].grid(True, alpha=0.3)
    
    # 4c. Histogram of blob assignments (middle frame)
    middle_frame = tracking_data[num_frames // 2]
    axes[1, 0].hist(middle_frame['blob_assignments'], bins=30, color='purple', edgecolor='black', alpha=0.7)
    axes[1, 0].set_xlabel('Blob Index', fontsize=11)
    axes[1, 0].set_ylabel('Number of Datapoints', fontsize=11)
    axes[1, 0].set_title(f'Datapoint Assignments Distribution (Frame {num_frames // 2})')
    axes[1, 0].grid(True, alpha=0.3, axis='y')
    
    # 4d. 3D scatter of datapoints colored by blob assignment (first frame)
    ax = fig.add_subplot(2, 2, 4, projection='3d')
    first_frame = tracking_data[0]
    positions = first_frame['datapoint_positions']
    assignments = first_frame['blob_assignments']
    
    scatter = ax.scatter(positions[:, 0], positions[:, 1], positions[:, 2], 
                        c=assignments, s=5, alpha=0.5, cmap='tab20')
    ax.set_xlabel('X', fontsize=10)
    ax.set_ylabel('Y', fontsize=10)
    ax.set_zlabel('Z', fontsize=10)
    ax.set_title('Datapoints Colored by Blob Assignment (Frame 0)')
    
    plt.tight_layout()
    plt.show()
    
    # ===== SUMMARY STATISTICS =====
    print(f"\n{'='*60}")
    print(f"TRACE VARIABLE SUMMARY FOR {stimulus.upper()}")
    print(f"{'='*60}")
    print(f"Total Frames: {num_frames}")
    print(f"Average Number of Blobs: {np.mean(num_blobs):.1f} (min: {min(num_blobs)}, max: {max(num_blobs)})")
    print(f"Number of Hyperblobs: {tracking_data[0]['n_hyperblobs']}")
    print(f"Number of Datapoints: {tracking_data[0]['n_datapoints']}")
    print(f"Average Datapoint Velocity: {np.mean(avg_datapoint_vel):.3f}")
    print(f"Average Blob Velocity: {np.mean(avg_vel_magnitudes):.3f}")
    print(f"{'='*60}")
    
else:
    print(f"No results found for {stimulus}")


In [ ]:
# 2D Hyperblob Assignment Images (First and Last Frame)
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.colors import ListedColormap
from PIL import Image

stimulus = VIDEO_NAME
if stimulus in all_results:
    tracking_data = all_results[stimulus][0]  # First seed run
    num_frames = len(tracking_data)
    
    # Get image dimensions from the actual data
    # The 3D motion data has been downsampled from the original video
    n_datapoints = tracking_data[0]['n_datapoints']
    
    # Standard DAVIS 3D motion extraction dimensions
    # Common resolutions: 520x960 = 499200
    if n_datapoints == 499200:
        img_height, img_width = 520, 960
    else:
        # Try to infer dimensions (assume standard aspect ratio)
        # Try common height/width ratios
        import math
        sqrt_n = int(math.sqrt(n_datapoints))
        for h in range(sqrt_n, sqrt_n + 1000):
            if n_datapoints % h == 0:
                w = n_datapoints // h
                if abs(w / h - 1.85) < 0.1:  # Close to 1920/1080 ratio
                    img_height, img_width = h, w
                    break
        else:
            # Fallback to square-ish
            img_height = img_width = sqrt_n
    
    print(f"Inferred image dimensions from data: {img_height}x{img_width} = {img_height*img_width} pixels")
    
    # Select only first and last frames
    frames_to_show = [0, num_frames - 1]
    
    fig, axes = plt.subplots(1, 2, figsize=(20, 8))
    fig.suptitle(f'{stimulus.capitalize()} Tracking - 2D Hyperblob Assignment Images', fontsize=18, fontweight='bold')
    
    # Create a colormap with distinct colors for each hyperblob + outliers
    n_hyperblobs = tracking_data[0]['n_hyperblobs']
    # Use tab10 for hyperblobs, add gray for outliers
    colors = plt.cm.tab10(np.linspace(0, 1, n_hyperblobs))
    colors = np.vstack([colors, [[0.5, 0.5, 0.5, 1.0]]])  # Add gray for outliers
    cmap = ListedColormap(colors)
    
    for idx, frame_idx in enumerate(frames_to_show):
        frame = tracking_data[frame_idx]
        
        # Get blob assignments for each datapoint
        blob_assignments = frame['blob_assignments']
        
        # Get hyperblob assignments for each blob
        hyperblob_assignments = frame['hyperblob_assignments']
        
        # Map each datapoint to its hyperblob
        # datapoint -> blob -> hyperblob
        # Handle out-of-bounds blob assignments (outliers) by assigning them to a special value
        n_blobs = len(hyperblob_assignments)
        valid_mask = blob_assignments < n_blobs
        
        datapoint_hyperblob_assignments = np.full(len(blob_assignments), n_hyperblobs, dtype=int)  # outliers get n_hyperblobs
        datapoint_hyperblob_assignments[valid_mask] = hyperblob_assignments[blob_assignments[valid_mask]]
        
        # Reshape assignments back to 2D image
        assignment_image = datapoint_hyperblob_assignments.reshape(img_height, img_width)
        
        # Count outliers
        n_outliers = np.sum(~valid_mask)
        
        # Display as image
        im = axes[idx].imshow(assignment_image, cmap=cmap, vmin=0, vmax=n_hyperblobs, interpolation='nearest')
        
        axes[idx].set_xlabel('X (pixels)', fontsize=12)
        axes[idx].set_ylabel('Y (pixels)', fontsize=12)
        axes[idx].set_title(f'Frame {frame_idx}\n{n_hyperblobs} hyperblobs, {n_outliers} outliers', 
                           fontsize=13, fontweight='bold')
        
        # Add colorbar for this subplot
        cbar = plt.colorbar(im, ax=axes[idx], fraction=0.046, pad=0.04)
        cbar.set_label('Hyperblob Index', fontsize=11)
        cbar.set_ticks(list(range(n_hyperblobs)) + [n_hyperblobs])
        cbar.set_ticklabels(list(range(n_hyperblobs)) + ['Outlier'])
    
    plt.tight_layout()
    plt.show()
    
    # Print statistics about hyperblob assignments for first and last frames
    print(f"\n{'='*60}")
    print(f"HYPERBLOB ASSIGNMENT STATISTICS FOR {stimulus.upper()}")
    print(f"{'='*60}")
    
    for frame_idx in frames_to_show:
        frame = tracking_data[frame_idx]
        blob_assignments = frame['blob_assignments']
        hyperblob_assignments = frame['hyperblob_assignments']
        
        # Handle out-of-bounds blob assignments (outliers)
        n_blobs = len(hyperblob_assignments)
        valid_mask = blob_assignments < n_blobs
        datapoint_hyperblob_assignments = np.full(len(blob_assignments), n_hyperblobs, dtype=int)
        datapoint_hyperblob_assignments[valid_mask] = hyperblob_assignments[blob_assignments[valid_mask]]
        
        unique, counts = np.unique(datapoint_hyperblob_assignments, return_counts=True)
        print(f"\nFrame {frame_idx}:")
        for hyperblob_idx, count in zip(unique, counts):
            percentage = (count / len(datapoint_hyperblob_assignments)) * 100
            label = f"Hyperblob {hyperblob_idx}" if hyperblob_idx < n_hyperblobs else "Outliers"
            print(f"  {label}: {count:6d} pixels ({percentage:5.1f}%)")
    
    print(f"{'='*60}")

else:
    print(f"No results found for {stimulus}")


In [ ]:
# Create binary segmentation video from tracking results
# Use the hyperblob that corresponds to the ground truth segmentation from frame 0

import cv2
from tqdm import tqdm

stimulus = VIDEO_NAME
if stimulus in all_results:
    tracking_data = all_results[stimulus][0]  # First seed run
    
    # Load the ground truth segmentation mask for frame 0
    first_frame_seg = get_segmentation_mask(stimulus, 0, DAVIS_SEGMASKS_PATH, img_dims=img_dims, flatten=True)
    
    # Find which hyperblob corresponds to the ground truth segmentation mask
    print("Finding hyperblob that matches ground truth segmentation...")
    frame0 = tracking_data[0]
    blob_assignments = frame0['blob_assignments']
    hyperblob_assignments = frame0['hyperblob_assignments']
    n_blobs = frame0['n_blobs']
    
    # Map datapoints to hyperblobs for frame 0
    valid_mask = blob_assignments < n_blobs
    datapoint_hyperblob_assignments = np.full(len(blob_assignments), -1, dtype=int)
    datapoint_hyperblob_assignments[valid_mask] = hyperblob_assignments[blob_assignments[valid_mask]]
    
    # Ensure first_frame_seg is flattened and matches the number of datapoints
    if len(first_frame_seg.shape) > 1:
        first_frame_seg_flat = first_frame_seg.flatten()
    else:
        first_frame_seg_flat = first_frame_seg
    
    # Verify dimensions match
    assert len(first_frame_seg_flat) == len(blob_assignments), \
        f"Segmentation mask size {len(first_frame_seg_flat)} doesn't match datapoints {len(blob_assignments)}"
    
    # Count overlap with ground truth segmentation mask
    # first_frame_seg is 1 for foreground, 0 for background
    hyperblob_overlaps = {}
    for hb_idx in range(frame0['n_hyperblobs']):
        hb_mask = datapoint_hyperblob_assignments == hb_idx
        overlap_with_gt = np.sum(hb_mask & (first_frame_seg_flat == 1))
        hyperblob_overlaps[hb_idx] = overlap_with_gt
    
    # Select hyperblob with maximum overlap
    HYPERBLOB_TO_SEGMENT = max(hyperblob_overlaps, key=hyperblob_overlaps.get)
    
    print(f"Hyperblob overlaps with ground truth (frame 0):")
    for hb in sorted(hyperblob_overlaps.keys()):
        print(f"  Hyperblob {hb}: {hyperblob_overlaps[hb]} pixels")
    print(f"\nSelected hyperblob {HYPERBLOB_TO_SEGMENT} (best match to ground truth)")
    
    # Configuration
    output_path = f"{VIDEO_NAME}_hyperblob_{HYPERBLOB_TO_SEGMENT}_segmentation_no_DINO.mp4"
    
    # Get dimensions
    n_datapoints = tracking_data[0]['n_datapoints']
    img_height, img_width = 520, 960  # Standard DAVIS resolution
    
    print(f"\nCreating binary segmentation video for hyperblob {HYPERBLOB_TO_SEGMENT}...")
    print(f"Resolution: {img_width}x{img_height}")
    
    # Try different codecs in order of preference
    codecs_to_try = [
        ('mp4v', 'MPEG-4'),
        ('XVID', 'XVID'),
        ('MJPG', 'Motion JPEG'),
    ]
    
    out = None
    for codec_str, codec_name in codecs_to_try:
        try:
            fourcc = cv2.VideoWriter_fourcc(*codec_str)
            out = cv2.VideoWriter(output_path, fourcc, 30.0, (img_width, img_height), isColor=True)
            if out.isOpened():
                print(f"Using codec: {codec_name} ({codec_str})")
                break
            out.release()
            out = None
        except:
            pass
    
    if out is None:
        raise RuntimeError("Could not initialize video writer with any codec")
    
    # Process each frame
    for frame_idx in tqdm(range(len(tracking_data)), desc="Rendering frames"):
        frame = tracking_data[frame_idx]
        blob_assignments = frame['blob_assignments']
        hyperblob_assignments = frame['hyperblob_assignments']
        n_blobs = frame['n_blobs']
        
        # Map datapoints to hyperblobs
        valid_mask = blob_assignments < n_blobs
        datapoint_hyperblob_assignments = np.full(len(blob_assignments), -1, dtype=int)
        datapoint_hyperblob_assignments[valid_mask] = hyperblob_assignments[blob_assignments[valid_mask]]
        
        # Create binary mask: hyperblob HYPERBLOB_TO_SEGMENT -> 255 (white), everything else -> 0 (black)
        binary_mask = np.where(datapoint_hyperblob_assignments == HYPERBLOB_TO_SEGMENT, 255, 0).astype(np.uint8)
        
        # Reshape to image
        binary_image = binary_mask.reshape(img_height, img_width)
        
        # Convert to BGR for video writer
        binary_bgr = cv2.cvtColor(binary_image, cv2.COLOR_GRAY2BGR)
        
        # Write frame
        out.write(binary_bgr)
    
    # Release video writer
    out.release()
    
    print(f"\nBinary segmentation video saved to: {output_path}")
    print(f"Total frames: {len(tracking_data)}")
    print(f"FPS: 30.0")
    
    # Show statistics
    total_pixels = img_height * img_width
    hyperblob_pixels_per_frame = []
    for frame in tracking_data:
        valid_mask = frame['blob_assignments'] < frame['n_blobs']
        valid_blob_assignments = frame['blob_assignments'][valid_mask]
        hyperblob_mask = frame['hyperblob_assignments'][valid_blob_assignments] == HYPERBLOB_TO_SEGMENT
        hyperblob_pixels_per_frame.append(np.sum(hyperblob_mask))
    
    avg_hyperblob_pixels = np.mean(hyperblob_pixels_per_frame)
    avg_background = total_pixels - avg_hyperblob_pixels
    
    print(f"\nHyperblob {HYPERBLOB_TO_SEGMENT} statistics:")
    print(f"  Average foreground pixels: {avg_hyperblob_pixels:.0f} ({100*avg_hyperblob_pixels/total_pixels:.1f}%)")
    print(f"  Average background pixels: {avg_background:.0f} ({100*avg_background/total_pixels:.1f}%)")
    
else:
    print(f"No results found for {stimulus}")

## Evaluation: False Positive and False Negative Rates

In [ ]:
# ============================================================================
# EVALUATION CONFIGURATION
# ============================================================================
BLOB_COUNTING_THRESHOLD = 100  # Minimum pixels for a blob to be counted in frame 0
                                # Only blobs with >= 100 assigned pixels in frame 0 are tracked

# Camera intrinsics for projection
fx = fy = 520.0
cx = img_dims[1] / 2.0  # img_dims = (height, width)
cy = img_dims[0] / 2.0

# ============================================================================
# Compute False Positive and False Negative Rates
# ============================================================================
# This evaluation measures blob means projected to 2D:
# - TRUE POSITIVES: Object blobs (frame 0) whose means project onto GT mask
# - FALSE NEGATIVES: Object blobs (frame 0) whose means project off GT mask (lost)
# - FALSE POSITIVES: Background blobs (frame 0) whose means project onto GT mask (contamination)

stimulus = VIDEO_NAME
if stimulus in all_results:
    tracking_data = all_results[stimulus][0]  # First seed run
    
    print("="*80)
    print("BLOB TRACKING EVALUATION WITH FALSE POSITIVE/NEGATIVE RATES")
    print("="*80)
    print(f"Configuration:")
    print(f"  Video: {VIDEO_NAME}")
    print(f"  Object hyperblob: {HYPERBLOB_TO_SEGMENT}")
    print(f"  Blob counting threshold: {BLOB_COUNTING_THRESHOLD} pixels")
    print(f"  Total frames: {len(tracking_data)}")
    print()

    # Get ground truth segmentation masks for all frames
    from genparticles.evaluation import get_segmentation_mask

    segmentation_masks = []
    for frame_idx in range(len(tracking_data)):
        seg_mask = get_segmentation_mask(
            stimulus, frame_idx, DAVIS_SEGMASKS_PATH, img_dims=img_dims, flatten=True
        )
        segmentation_masks.append(seg_mask)

    print(f"Loaded {len(segmentation_masks)} ground truth segmentation masks\n")

    # ============================================================================
    # Step 1: Identify reference blobs in frame 0
    # ============================================================================
    frame0 = tracking_data[0]
    blob_assignments_frame0 = frame0['blob_assignments']
    hyperblob_assignments_frame0 = frame0['hyperblob_assignments']
    blob_means_frame0 = frame0['blob_means']
    n_blobs_frame0 = frame0['n_blobs']
    gt_mask_frame0 = segmentation_masks[0]

    # Count how many pixels each blob has in frame 0
    blob_pixel_counts_frame0 = np.bincount(
        blob_assignments_frame0[blob_assignments_frame0 < n_blobs_frame0],
        minlength=n_blobs_frame0
    )

    # Filter blobs: only keep those with >= BLOB_COUNTING_THRESHOLD pixels in frame 0
    significant_blobs = np.where(blob_pixel_counts_frame0 >= BLOB_COUNTING_THRESHOLD)[0]

    # Classify significant blobs as object or background based on hyperblob assignment
    object_blobs_frame0 = []
    background_blobs_frame0 = []

    for blob_idx in significant_blobs:
        if hyperblob_assignments_frame0[blob_idx] == HYPERBLOB_TO_SEGMENT:
            object_blobs_frame0.append(blob_idx)
        else:
            background_blobs_frame0.append(blob_idx)

    object_blobs_frame0 = np.array(object_blobs_frame0)
    background_blobs_frame0 = np.array(background_blobs_frame0)

    print(f"Frame 0 blob classification (threshold={BLOB_COUNTING_THRESHOLD} pixels):")
    print(f"  Total significant blobs (particles): {len(significant_blobs)}")
    print(f"  Object blobs (hyperblob {HYPERBLOB_TO_SEGMENT}): {len(object_blobs_frame0)}")
    print(f"  Background blobs (other hyperblobs): {len(background_blobs_frame0)}")
    print()

    # Verify object blobs by projecting their means to GT mask
    # Project blob means to 2D pixel coordinates
    x_2d = (blob_means_frame0[:, 0] / (blob_means_frame0[:, 2] + 1e-8)) * fx + cx
    y_2d = (blob_means_frame0[:, 1] / (blob_means_frame0[:, 2] + 1e-8)) * fy + cy
    x_2d = np.clip(x_2d.astype(int), 0, img_dims[1] - 1)
    y_2d = np.clip(y_2d.astype(int), 0, img_dims[0] - 1)
    pixel_indices_frame0 = y_2d * img_dims[1] + x_2d

    # Check which object blobs have means on GT mask
    object_blobs_on_mask = []
    for blob_idx in object_blobs_frame0:
        pixel_idx = pixel_indices_frame0[blob_idx]
        if pixel_idx < len(gt_mask_frame0) and gt_mask_frame0[pixel_idx]:
            object_blobs_on_mask.append(blob_idx)

    # Check which background blobs have means off GT mask
    background_blobs_off_mask = []
    for blob_idx in background_blobs_frame0:
        pixel_idx = pixel_indices_frame0[blob_idx]
        if pixel_idx >= len(gt_mask_frame0) or not gt_mask_frame0[pixel_idx]:
            background_blobs_off_mask.append(blob_idx)

    object_blobs_on_mask = np.array(object_blobs_on_mask)
    background_blobs_off_mask = np.array(background_blobs_off_mask)

    n_object_blobs = len(object_blobs_on_mask)
    n_background_blobs = len(background_blobs_off_mask)

    print(f"Reference blobs for tracking (verified against GT mask in frame 0):")
    print(f"  Object blobs with means ON GT mask: {n_object_blobs}")
    print(f"  Background blobs with means OFF GT mask: {n_background_blobs}")
    print()

    # ============================================================================
    # Step 2: Track blob means over time
    # ============================================================================
    # For each frame, project blob means and check if they're on/off GT mask

    false_negative_rates = []  # Object blobs that fell off GT mask
    false_positive_rates = []  # Background blobs that moved onto GT mask
    true_positive_counts = []  # Object blobs still on GT mask
    false_negative_counts = []  # Object blobs lost
    false_positive_counts = []  # Background blobs contaminating

    for frame_idx in range(len(tracking_data)):
        frame = tracking_data[frame_idx]
        blob_means = frame['blob_means']
        n_blobs = frame['n_blobs']
        gt_mask = segmentation_masks[frame_idx]
        
        # Project blob means to 2D
        x_2d = (blob_means[:, 0] / (blob_means[:, 2] + 1e-8)) * fx + cx
        y_2d = (blob_means[:, 1] / (blob_means[:, 2] + 1e-8)) * fy + cy
        x_2d = np.clip(x_2d.astype(int), 0, img_dims[1] - 1)
        y_2d = np.clip(y_2d.astype(int), 0, img_dims[0] - 1)
        pixel_indices = y_2d * img_dims[1] + x_2d
        
        # Check object blobs: how many still have means on GT mask?
        tp_count = 0
        for blob_idx in object_blobs_on_mask:
            if blob_idx < n_blobs:  # Blob still exists
                pixel_idx = pixel_indices[blob_idx]
                if pixel_idx < len(gt_mask) and gt_mask[pixel_idx]:
                    tp_count += 1
        
        # FALSE NEGATIVES: Object blobs that fell off GT mask
        fn_count = n_object_blobs - tp_count
        fn_rate = (fn_count / n_object_blobs) * 100 if n_object_blobs > 0 else 0.0
        
        # Check background blobs: how many now have means on GT mask?
        fp_count = 0
        for blob_idx in background_blobs_off_mask:
            if blob_idx < n_blobs:  # Blob still exists
                pixel_idx = pixel_indices[blob_idx]
                if pixel_idx < len(gt_mask) and gt_mask[pixel_idx]:
                    fp_count += 1
        
        # FALSE POSITIVES: Background blobs that moved onto GT mask
        fp_rate = (fp_count / n_background_blobs) * 100 if n_background_blobs > 0 else 0.0
        
        false_negative_rates.append(fn_rate)
        false_positive_rates.append(fp_rate)
        true_positive_counts.append(tp_count)
        false_negative_counts.append(fn_count)
        false_positive_counts.append(fp_count)

    # ============================================================================
    # Step 3: Summary statistics
    # ============================================================================
    mean_fn_rate = np.mean(false_negative_rates)
    mean_fp_rate = np.mean(false_positive_rates)
    mean_tp = np.mean(true_positive_counts)
    mean_fn = np.mean(false_negative_counts)
    mean_fp = np.mean(false_positive_counts)

    print("="*80)
    print("RESULTS")
    print("="*80)
    print(f"Mean False Negative Rate: {mean_fn_rate:.2f}%")
    print(f"  (Object blobs whose means fell off GT mask)")
    print(f"  Average lost: {mean_fn:.1f} / {n_object_blobs} blobs")
    print()
    print(f"Mean False Positive Rate: {mean_fp_rate:.2f}%")
    print(f"  (Background blobs whose means moved onto GT mask)")
    print(f"  Average contamination: {mean_fp:.1f} / {n_background_blobs} blobs")
    print()
    print(f"Mean True Positive Count: {mean_tp:.1f} / {n_object_blobs} blobs")
    print(f"  (Object blobs retained on GT mask)")
    print()
    print(f"Interpretation:")
    print(f"  - Lower FN rate = Better object blob retention")
    print(f"  - Lower FP rate = Less background blob contamination")
    print(f"  - Perfect tracking: FN=0%, FP=0%")
    print(f"  - Only blobs with ≥{BLOB_COUNTING_THRESHOLD} pixels in frame 0 are tracked")
    print("="*80)

    # Show per-frame breakdown (every 10 frames)
    print(f"\nPer-frame breakdown (every 10 frames):")
    print(f"{'Frame':<8} {'TP':<8} {'FN':<8} {'FP':<8} {'FN Rate (%)':<15} {'FP Rate (%)':<15}")
    print("-" * 70)
    for i in range(0, len(false_negative_rates), 10):
        print(f"{i:<8} {true_positive_counts[i]:<8} {false_negative_counts[i]:<8} {false_positive_counts[i]:<8} "
              f"{false_negative_rates[i]:<15.2f} {false_positive_rates[i]:<15.2f}")

    # ============================================================================
    # Step 4: Visualizations
    # ============================================================================
    import matplotlib.pyplot as plt

    fig, axes = plt.subplots(2, 1, figsize=(14, 10))
    fig.suptitle(f'{VIDEO_NAME.capitalize()} - Blob Tracking: False Positive and False Negative Rates', 
                 fontsize=16, fontweight='bold')

    # Plot False Negative Rate
    axes[0].plot(range(len(false_negative_rates)), false_negative_rates, 
                 linewidth=2, color='red', label='False Negative Rate')
    axes[0].axhline(y=mean_fn_rate, color='darkred', linestyle='--', linewidth=2, 
                    label=f'Mean: {mean_fn_rate:.2f}%')
    axes[0].set_xlabel('Frame', fontsize=12)
    axes[0].set_ylabel('False Negative Rate (%)', fontsize=12)
    axes[0].set_title('Object Blobs Lost (means fell off GT mask)', fontsize=14)
    axes[0].grid(True, alpha=0.3)
    axes[0].legend(fontsize=10)
    axes[0].set_ylim([0, 105])

    # Plot False Positive Rate
    axes[1].plot(range(len(false_positive_rates)), false_positive_rates, 
                 linewidth=2, color='blue', label='False Positive Rate')
    axes[1].axhline(y=mean_fp_rate, color='darkblue', linestyle='--', linewidth=2, 
                    label=f'Mean: {mean_fp_rate:.2f}%')
    axes[1].set_xlabel('Frame', fontsize=12)
    axes[1].set_ylabel('False Positive Rate (%)', fontsize=12)
    axes[1].set_title('Background Blobs Contaminating (means moved onto GT mask)', fontsize=14)
    axes[1].grid(True, alpha=0.3)
    axes[1].legend(fontsize=10)
    axes[1].set_ylim([0, 105])

    plt.tight_layout()
    plt.show()

    # Combined plot
    fig, ax = plt.subplots(figsize=(14, 6))
    ax.plot(range(len(false_negative_rates)), false_negative_rates, 
            linewidth=2, color='red', label=f'False Negative Rate (mean: {mean_fn_rate:.2f}%)', alpha=0.8)
    ax.plot(range(len(false_positive_rates)), false_positive_rates, 
            linewidth=2, color='blue', label=f'False Positive Rate (mean: {mean_fp_rate:.2f}%)', alpha=0.8)
    ax.set_xlabel('Frame', fontsize=12)
    ax.set_ylabel('Rate (%)', fontsize=12)
    ax.set_title(f'{VIDEO_NAME.capitalize()} - Blob Tracking Error Rates Over Time', 
                 fontsize=14, fontweight='bold')
    ax.grid(True, alpha=0.3)
    ax.legend(fontsize=11)
    ax.set_ylim([0, 105])
    plt.tight_layout()
    plt.show()

    print("\n" + "="*80)
    print(f"Evaluation complete!")
    print("="*80)
    
else:
    print(f"No results found for {stimulus}")